"""
shape_pca_permanova_classification
Scale-independent shape descriptors: outlier screening, PCA, PERMANOVA/PERMDISP,
pairwise tests and Random Forest classification (shape, EFD, combined).
 
Inputs : basic_shape_features.csv      (465 quality-controlled leaves)
         77harmonics_normalized.csv    (written by efd_analysis; 442 leaves)
Outputs: Table I (desc_table.csv), Figure 2, Figure 3, Table II (pairwise_permanova_table.csv),
         Table IV (confusion_matrix_combined.csv), classification_summary.csv, shape_features_clean_442.csv
"""

In [2]:
import itertools
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.spatial.distance import mahalanobis, pdist, squareform
from scipy.stats import chi2
from skbio.stats.distance import DistanceMatrix, permanova, permdisp
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.multitest import multipletests
 
SEED = 42
SHAPE_COLS = ["aspect_ratio", "extent", "roundness", "circularity", "convexity", "solidity"]
COUNTRY_ORDER = ["BH", "OM", "QA", "SA", "UAE"]

In [3]:
# ============================================================ 1. load + multivariate outlier screening
df = pd.read_csv("basic_shape_features.csv")
X_all = StandardScaler().fit_transform(df[SHAPE_COLS])
inv_cov = np.linalg.inv(np.cov(X_all, rowvar=False))
mean_vec = X_all.mean(axis=0)
df["mahal_dist"] = [mahalanobis(x, mean_vec, inv_cov) for x in X_all]
threshold = np.sqrt(chi2.ppf(1 - 0.001, df=len(SHAPE_COLS)))      # chi-square, alpha = 0.001, 6 df
df_clean = df[df["mahal_dist"] < threshold].copy()
 
print(f"Leaves: {len(df)} -> removed {len(df) - len(df_clean)} -> final {len(df_clean)}")
assert len(df_clean) == 442
print(df_clean["country"].value_counts())
df_clean.to_csv("shape_features_clean_442.csv", index=False)
 
labels = df_clean["country"].values
X_std = StandardScaler().fit_transform(df_clean[SHAPE_COLS])      # standardised once; reused below

Leaves: 465 -> removed 23 -> final 442
country
BH     170
UAE    142
SA      53
QA      52
OM      25
Name: count, dtype: int64


In [4]:
# ============================================================ 2. Table I and Figure 2
desc = df_clean[SHAPE_COLS].describe().T[["mean", "std", "min", "max"]].round(2)
desc.to_csv("desc_table.csv")
print(desc)
 
corr = df_clean[SHAPE_COLS].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f", square=True)
plt.title("Correlation Matrix of Scale-Independent Shape Descriptors")
plt.savefig("Figure2_correlation_matrix.png", dpi=300, bbox_inches="tight")
plt.close()

              mean   std   min   max
aspect_ratio  2.30  0.63  1.16  4.97
extent        0.60  0.10  0.25  0.83
roundness     0.44  0.12  0.17  0.83
circularity   0.61  0.10  0.32  0.86
convexity     0.93  0.02  0.79  0.97
solidity      0.95  0.04  0.75  1.00


In [5]:
# ============================================================ 3. PCA, centroids, Figure 3
pca = PCA()
scores = pca.fit_transform(X_std)
print("Explained variance (%):", (pca.explained_variance_ratio_ * 100).round(2))
print("Cumulative (%):", (np.cumsum(pca.explained_variance_ratio_) * 100).round(2))
print(pd.DataFrame(pca.components_.T, index=SHAPE_COLS,
                   columns=[f"PC{i + 1}" for i in range(len(SHAPE_COLS))]).round(3))
 
pca_df = pd.DataFrame(scores[:, :3], columns=["PC1", "PC2", "PC3"], index=df_clean.index)
pca_df["country"] = labels
centroids = pca_df.groupby("country")[["PC1", "PC2", "PC3"]].mean().loc[COUNTRY_ORDER]
print(centroids.round(3))
centroid_dist = pd.DataFrame(squareform(pdist(centroids.values)), index=COUNTRY_ORDER, columns=COUNTRY_ORDER)
centroid_dist.round(2).to_csv("country_centroid_distances.csv")
print(centroid_dist.round(2))
 
plt.figure(figsize=(8, 6))
for country in ["BH", "QA", "OM", "SA", "UAE"]:
    sub = pca_df[pca_df["country"] == country]
    plt.scatter(sub["PC1"], sub["PC2"], s=30, alpha=0.6, label=country)
for country in ["BH", "QA", "OM", "SA", "UAE"]:
    cx, cy = centroids.loc[country, ["PC1", "PC2"]]
    plt.scatter(cx, cy, s=180, marker="X", edgecolor="black", linewidth=1.2)
    plt.text(cx + 0.08, cy + 0.08, country, fontsize=10, weight="bold")
plt.axhline(0, linewidth=0.8, color="gray", linestyle="--")
plt.axvline(0, linewidth=0.8, color="gray", linestyle="--")
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0] * 100:.2f}%)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1] * 100:.2f}%)")
plt.title("PCA Scatter Plot of Scale-Independent Leaf Shape Descriptors")
plt.legend(title="Country")
plt.tight_layout()
plt.savefig("Figure3_PCA_PC1_PC2_with_centroids.png", dpi=300, bbox_inches="tight")
plt.close()

Explained variance (%): [59.4  25.91  7.65  5.45  1.12  0.48]
Cumulative (%): [ 59.4   85.3   92.96  98.41  99.52 100.  ]
                PC1    PC2    PC3    PC4    PC5    PC6
aspect_ratio -0.457  0.337  0.315  0.010  0.674  0.352
extent        0.432 -0.030  0.713 -0.548  0.009 -0.050
roundness     0.482 -0.262 -0.210  0.162  0.723 -0.325
circularity   0.512  0.101 -0.253 -0.005 -0.010  0.814
convexity     0.101  0.723 -0.409 -0.465  0.053 -0.283
solidity      0.316  0.532  0.341  0.676 -0.140 -0.156
           PC1    PC2    PC3
country                     
BH       0.559 -0.534 -0.072
OM       0.560  0.279 -0.035
QA      -0.150  0.267  0.156
SA      -0.958  0.565  0.172
UAE     -0.356  0.281 -0.030
       BH    OM    QA    SA   UAE
BH   0.00  0.81  1.09  1.89  1.23
OM   0.81  0.00  0.74  1.56  0.92
QA   1.09  0.74  0.00  0.86  0.28
SA   1.89  1.56  0.86  0.00  0.70
UAE  1.23  0.92  0.28  0.70  0.00


In [6]:
# ============================================================ 4. PERMANOVA, PERMDISP, pairwise (Table II)
def make_dm(X):
    ids = [str(i) for i in range(len(X))]
    return DistanceMatrix(squareform(pdist(X, metric="euclidean")), ids=ids), ids
 
 
def r_squared(F, n, k):
    return F * (k - 1) / (F * (k - 1) + (n - k))     # scikit-bio does not return R2
 
 
dm, ids = make_dm(X_std)
grouping = pd.Series(labels, index=ids, name="country")
 
res = permanova(dm, grouping, permutations=999, seed=SEED)
R2 = r_squared(res["test statistic"], len(grouping), grouping.nunique())
print(f"PERMANOVA: pseudo-F = {res['test statistic']:.2f}, p = {res['p-value']:.3f}, R2 = {R2:.4f}")
print("PERMDISP:\n", permdisp(dm, grouping, permutations=999, seed=SEED))
 
rows = []
for c1, c2 in itertools.combinations(sorted(set(labels)), 2):
    mask = np.isin(labels, [c1, c2])
    dm_p, ids_p = make_dm(X_std[mask])
    res_p = permanova(dm_p, pd.Series(labels[mask], index=ids_p, name="country"),
                      permutations=999, seed=SEED)
    rows.append({"Country_1": c1, "Country_2": c2, "N": int(mask.sum()),
                 "Pseudo_F": res_p["test statistic"],
                 "R2": r_squared(res_p["test statistic"], int(mask.sum()), 2),
                 "p_raw": res_p["p-value"]})
pairwise = pd.DataFrame(rows)
pairwise["Significant_FDR"], pairwise["p_FDR"], _, _ = multipletests(pairwise["p_raw"], alpha=0.05, method="fdr_bh")
pairwise.round({"Pseudo_F": 2, "R2": 3, "p_raw": 3, "p_FDR": 3}).to_csv("pairwise_permanova_table.csv", index=False)
print(pairwise.round(3).to_string(index=False))

PERMANOVA: pseudo-F = 9.81, p = 0.001, R2 = 0.0824


/Users/mustafamuhamed/morpho_env/lib/python3.10/site-packages/skbio/stats/ordination/_principal_coordinate_analysis.py:164: RuntimeWarning: EIGH: since no value for dimensions is specified, PCoA for all dimensions will be computed, which may result in long computation time if the original distance matrix is large.
  warn(


PERMDISP:
 method name               PERMDISP
test statistic name        F-value
sample size                    442
number of groups                 5
test statistic            0.529122
p-value                      0.718
number of permutations         999
Name: PERMDISP results, dtype: object
Country_1 Country_2   N  Pseudo_F    R2  p_raw  Significant_FDR  p_FDR
       BH        OM 195     2.467 0.013  0.094            False  0.118
       BH        QA 222     8.581 0.038  0.001             True  0.003
       BH        SA 223    24.336 0.099  0.001             True  0.003
       BH       UAE 312    20.895 0.063  0.001             True  0.003
       OM        QA  77     1.948 0.025  0.129            False  0.143
       OM        SA  78     7.372 0.088  0.002             True  0.005
       OM       UAE 167     3.426 0.020  0.031             True  0.044
       QA        SA 105     3.781 0.035  0.022             True  0.037
       QA       UAE 194     0.588 0.003  0.598            False  0.

In [7]:
# ============================================================ 5. Random Forest classification
# --- align shape features with the 77 EFD coefficients by leaf ID
shape = df_clean.assign(leaf_id=df_clean["filename"].str.replace(".png", "", regex=False)).set_index("leaf_id")
efd = pd.read_csv("77harmonics_normalized.csv", index_col="leaf_id").drop(columns=["country"])
efd.columns = [f"EFD_{i + 1}" for i in range(efd.shape[1])]
 
common_ids = shape.index.intersection(efd.index)
assert len(common_ids) == 442, f"leaf IDs do not align ({len(common_ids)} common)"
shape_aligned = shape.loc[common_ids, SHAPE_COLS]
efd_aligned = efd.loc[common_ids]
y = shape.loc[common_ids, "country"]
 
# --- one stratified 80/20 split, test set untouched until final evaluation
train_ids, test_ids = train_test_split(common_ids, test_size=0.20, stratify=y.loc[common_ids], random_state=SEED)
y_train, y_test = y.loc[train_ids], y.loc[test_ids]
print("Train:", len(train_ids), "| Test:", len(test_ids))
print(y_test.value_counts())
 
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
 
 
def rf():
    return RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=SEED)
 
 
def efd_block():
    # scaler + PCA are fitted inside every training fold -> no information leakage
    return Pipeline([("scaler", StandardScaler()), ("pca", PCA(n_components=20))])
 
 
efd_cols = efd_aligned.columns.tolist()
models = {
    "Morphometric (6 descriptors)": (shape_aligned, rf()),
    "EFD (77 coeff. -> 20 PCs)": (efd_aligned, Pipeline([("scaler", StandardScaler()),
                                                         ("pca", PCA(n_components=20)),
                                                         ("rf", rf())])),
    "Combined": (pd.concat([shape_aligned, efd_aligned], axis=1),
                 Pipeline([("features", ColumnTransformer([("shape", "passthrough", SHAPE_COLS),
                                                           ("efd", efd_block(), efd_cols)])),
                           ("rf", rf())])),
}
 
summary = []
for name, (X, model) in models.items():
    cv_res = cross_validate(model, X.loc[train_ids], y_train, cv=cv,
                            scoring={"accuracy": "accuracy", "f1_macro": "f1_macro"})
    model.fit(X.loc[train_ids], y_train)
    pred = model.predict(X.loc[test_ids])
    summary.append({"Model": name,
                    "CV_accuracy_mean": cv_res["test_accuracy"].mean(),
                    "CV_accuracy_sd": cv_res["test_accuracy"].std(),
                    "CV_macroF1_mean": cv_res["test_f1_macro"].mean(),
                    "Test_accuracy": accuracy_score(y_test, pred),
                    "Test_macroF1": f1_score(y_test, pred, average="macro")})
    print(f"\n===== RF - {name} =====")
    print(classification_report(y_test, pred, digits=3, zero_division=0))
    if name == "Combined":
        cm = pd.DataFrame(confusion_matrix(y_test, pred, labels=COUNTRY_ORDER),
                          index=COUNTRY_ORDER, columns=COUNTRY_ORDER)
        cm.to_csv("confusion_matrix_combined.csv")          # Table IV
        print(cm)
 
pd.DataFrame(summary).round(4).to_csv("classification_summary.csv", index=False)
print(pd.DataFrame(summary).round(3).to_string(index=False))

Train: 353 | Test: 89
country
BH     34
UAE    29
SA     11
QA     10
OM      5
Name: count, dtype: int64

===== RF - Morphometric (6 descriptors) =====
              precision    recall  f1-score   support

          BH      0.578     0.765     0.658        34
          OM      0.000     0.000     0.000         5
          QA      0.500     0.200     0.286        10
          SA      0.400     0.182     0.250        11
         UAE      0.406     0.448     0.426        29

    accuracy                          0.483        89
   macro avg      0.377     0.319     0.324        89
weighted avg      0.459     0.483     0.453        89


===== RF - EFD (77 coeff. -> 20 PCs) =====
              precision    recall  f1-score   support

          BH      0.558     0.853     0.674        34
          OM      0.000     0.000     0.000         5
          QA      0.667     0.200     0.308        10
          SA      0.500     0.091     0.154        11
         UAE      0.500     0.552     0.525